# 🟤 Bronze Delta Ingest

Formalizes the Bronze layer as a real **Delta table** instead of reading the
raw CSV glob directly on every run.

**Why:** `bronze_to_silver_v2.ipynb` currently reads
`{CSV_ROOT}/**/*.csv` fresh every time it runs — there's no single frozen,
schema-enforced, versioned artifact in between the parser output and Silver.
This notebook adds that: a managed Delta table with the same columns as the
CSVs, plus provenance (`_source_file`, `_ingest_ts`, `_batch_id`).

**Source:** CSVs produced by `pdf_to_csv.ipynb`
(`/Volumes/rankrangers_project_data/pdf/cet_raw_pdfs/data/mahacet_cutoffs_2025_csv`)
**Output:** `rankrangers_project_data.bronze.mhcet_allotments_raw`

**Downstream change:** once this table exists and is verified, point
`bronze_to_silver_v2.ipynb` Cell 2 at it instead of the CSV glob (one-line
change — see that notebook's Cell 2).

## Cell 1 — Config

In [ ]:
import uuid

CSV_ROOT   = '/Volumes/rankrangers_project_data/pdf/cet_raw_pdfs/data/mahacet_cutoffs_2025_csv'
CATALOG    = 'rankrangers_project_data'
SCHEMA     = 'bronze'
TABLE      = 'mhcet_allotments_raw'
FULL_TABLE = f'{CATALOG}.{SCHEMA}.{TABLE}'

# One identifier per run of this notebook, stamped onto every row it writes -
# lets you trace which ingestion run produced which rows via _batch_id.
BATCH_ID = str(uuid.uuid4())

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}')
print(f'Source  : {CSV_ROOT}')
print(f'Target  : {FULL_TABLE}')
print(f'Batch ID: {BATCH_ID}')

## Cell 2 — Load Raw CSVs + Add Provenance Columns

Same read options `bronze_to_silver_v2.ipynb` Cell 2 uses (header,
inferSchema, UTF-8) — this table should be a drop-in replacement for that
read, not a different shape of data.

In [ ]:
from pyspark.sql import functions as F

df = (spark.read
      .option('header', True)
      .option('inferSchema', True)
      .option('encoding', 'UTF-8')
      .csv(f'{CSV_ROOT}/**/*.csv')
      .withColumn('_source_file', F.input_file_name())
      .withColumn('_ingest_ts', F.current_timestamp())
      .withColumn('_batch_id', F.lit(BATCH_ID)))

raw_count = df.count()
print(f'Raw rows loaded: {raw_count:,}')

## Cell 3 — Sanity Check

`bronze_to_silver_v2.ipynb`'s own markdown cites **735,136 rows** across
1,480 CSVs as the raw count it scanned to build v2 — this should match (or
be explainable if it doesn't, e.g. if new CSVs were added/regenerated since).

In [ ]:
EXPECTED_RAW_ROWS = 735_136

if raw_count != EXPECTED_RAW_ROWS:
    print(f'⚠️  Raw row count ({raw_count:,}) does not match the '
          f'{EXPECTED_RAW_ROWS:,} rows bronze_to_silver_v2.ipynb was built '
          f'against — confirm whether the CSV source changed before '
          f'proceeding.')
else:
    print(f'✅ Raw row count matches the {EXPECTED_RAW_ROWS:,} baseline.')

file_count = df.select('_source_file').distinct().count()
print(f'Distinct source files: {file_count:,} (expected 1,480)')

## Cell 4 — Write Bronze Delta Table

Uses `overwrite` (same convention as `bronze_to_silver_v2.ipynb` /
`silver_to_gold_v2.ipynb`) rather than `append`, so re-running this notebook
doesn't duplicate rows. Delta's own version history (`DESCRIBE HISTORY`)
already gives you time-travel across runs — you don't need append-only
writes to get that.

In [ ]:
df.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', True) \
    .saveAsTable(FULL_TABLE)

written = spark.table(FULL_TABLE).count()
print(f'✅ Written to {FULL_TABLE}')
print(f'   Total rows: {written:,}')

## Cell 5 — Quick Verify

In [ ]:
display(spark.sql(f'''
    SELECT cap_round, COUNT(*) AS rows, COUNT(DISTINCT _source_file) AS files
    FROM {FULL_TABLE}
    GROUP BY cap_round
    ORDER BY cap_round
'''))

display(spark.sql(f'DESCRIBE HISTORY {FULL_TABLE}').select(
    'version', 'timestamp', 'operation', 'operationMetrics'
))